### Step 1: Install Dependencies

In [ ]:
! pip install torch transformers datasets accelerate seqeval

> You **MUST** work with GPU, so check first: 

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
else:
    print("Please switch to a GPU-enabled environment.")

### Step 2: Import Libraries

In [ ]:
from collections import Counter

from datasets import load_datase
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import precision_score, recall_score, f1_score
from torch.quantization import quantize_dynamic

### Step 3: Dataset Preparation

#### 3.1: Load Data

In [ ]:
# Load "lhoestq/conll2003" dataset using HuggingFace datasets library
dataset = 

#### 3.2: Show Data

In [ ]:
# Show samples of data before start working


In [ ]:
# Check Data Balancing


#### 3.3: Data Formatting
> Convert dataset into BERT-compatible format
<br>FROM:<br>
{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}
 <br><br>
 TO:<br>
 {'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [101,
  7270,
  22961,
  1528,
  1840,
  1106,
  21423,
  1418,
  2495,
  12913,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]}

Where:
- input_ids will be the X and labels will be the Y
- 101 token id is the <sos> token
- 102 token id is the <eos> token
- -100 is the label for <sos> and <eos>

In [ ]:
# use the following tokenizer "bert-base-cased"
tokenizer = 

### Step 4: Model Loading

In [ ]:
# Load pre-trained BERT model with a token classification head
model = 

In [ ]:
# Check the model architecture


### Step 5: Fine-Tuning

#### 5.1: Hyperparameter Selection

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir="./logs",
    logging_steps=10,
)

#### 5.2: Evaluation Metrices

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    true_labels = [[id2label[l] for l in label] for label in labels]
    true_preds = [[id2label[p] for p in pred] for pred in preds]
    
    precision = precision_score(true_labels, true_preds)
    recall = recall_score(true_labels, true_preds)
    f1 = f1_score(true_labels, true_preds)
    
    return {"precision": precision, "recall": recall, "f1": f1}

#### 5.3: Training Loop
> ONLY Choose one of the following methods NOT BOTH

In [ ]:
# Default Standard Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,  # Custom function to calculate F1, precision, recall
)

trainer.train()

In [ ]:
# OR Custom Trainer
from torch.utils.data import DataLoader
from transformers import AdamW

train_dataloader = DataLoader(tokenized_dataset["train"], 
                              batch_size=16, 
                              shuffle=True)
optimizer = AdamW(model.parameters(), 
                  lr=2e-5)

for epoch in range(3):  # Number of epochs
    model.train()
    for batch in train_dataloader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

### Step 6: Save Weights

In [ ]:
# Save the model and tokenizer
model.save_pretrained("./ner_model")
tokenizer.save_pretrained("./ner_model")

print("Model and tokenizer saved to ./ner_model")

### Step 7: Predict & Test

In [ ]:
def predict(text: str):
    tokens = tokenizer(text, 
                       return_tensors="pt", 
                       truncation=True, 
                       is_split_into_words=True)
    with torch.no_grad():
        outputs = model(**tokens)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
    return {"tokens": tokenizer.tokenize(text), "predictions": predictions}

### Step 8: Model Quantization

In [ ]:
quantized_model = quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
quantized_model.save_pretrained("./quantized_ner_model")
print("Quantized model saved.")

In [ ]:
def predict(text: str):
    tokens = tokenizer(text, 
                       return_tensors="pt", 
                       truncation=True, 
                       is_split_into_words=True)
    with torch.no_grad():
        outputs = quantized_model(**tokens)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
    return {"tokens": tokenizer.tokenize(text), "predictions": predictions}

### Step 9: Comparison and Conclusion

In [ ]:
# Compare the trained & quantized models' predictions and give a comment
# Compare the time taken for both models using time library and give a comment


> Thanls a lot for your Effort